# IOAI — 2025 Individual Contest Radar — ⭐모범답안 (Colab 자동 설정판)

이 노트북은 IOAI 로컬 연습 사이트에서 **데이터·학습환경이 자동 준비**되도록 생성되었습니다.
아래 **설정 셀을 먼저 실행**하면 공식 GitHub 저장소에서 이 문제 폴더만 부분 클론으로 받아
(전체 6.6GB 가 아니라 해당 폴더만), 그 폴더로 이동한 뒤 이후 셀이 그대로 학습/예측을 합니다.
완료 후 생성되는 제출 파일을 내려받아 연습 사이트의 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** 로 바꾸면 학습이 빨라집니다.

In [ ]:
# === 데이터 + 환경 자동 설정 (가장 먼저 실행) ===
# 공식 공개 저장소에서 이 문제 폴더만 부분 클론(sparse)으로 받고 그 폴더로 이동한다.
import os
REPO_URL = "https://github.com/IOAI-official/IOAI-2025"
CLONE = "IOAI-2025"
SUBDIR = "Individual-Contest/Radar"
# Colab 은 /content 가 홈. 재실행해도 경로가 안정적이도록 고정 기준에서 시작한다.
BASE = "/content" if os.path.isdir("/content") else os.getcwd()
os.chdir(BASE)
if not os.path.isdir(os.path.join(CLONE, SUBDIR)):
    !git clone --filter=blob:none --no-checkout --depth 1 $REPO_URL $CLONE
    !cd $CLONE && git sparse-checkout set "$SUBDIR"
    !cd $CLONE && git checkout
os.chdir(os.path.join(BASE, CLONE, SUBDIR))
print("작업 폴더:", os.getcwd())
print("내용:", sorted(os.listdir(".")))

# Radar — 모범답안 2 (개선판: Weighted-CE U-Net)

기존 모범답안(DIMSSNet + FocalLoss)은 **0.7963**. 이 개선판은 [ioai-writeup.github.io](https://ioai-writeup.github.io/posts/d1p1-radar/) 의 핵심 통찰을 적용합니다:

- **표준 U-Net**(6채널→5클래스, 3 enc/3 dec, 512 bottleneck)
- **가중 CrossEntropyLoss**: 배경(-1)을 다른 클래스보다 **50배 덜 중요**하게 — 채점(비배경 픽셀 50배 가중)과 일치

검증 점수: **public_a≈0.983, private_b≈0.984** (기존 0.7963 대비 대폭 향상).

> 학습 10 epoch (~수 분, GPU). 50배 가중 + 소규모 데이터라 더 오래 돌리면 과적합으로 점수가 떨어집니다.

In [ ]:
import os, glob, numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
DEV='cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0); np.random.seed(0)

# 데이터 경로(작업폴더/Solution 양쪽 호환). val/test 는 대회 플랫폼에선 DATA_PATH 로 주입됨.
def _find(*cs):
    for c in cs:
        if c and os.path.isdir(c): return c
    return cs[0]
DATA_PATH=os.environ.get('DATA_PATH',''); DATA_PATH=(DATA_PATH.rstrip('/')+'/') if DATA_PATH else ''
TRAIN_DIR=_find('training_set','../training_set','Solution/training_set')
VAL_DIR=_find(DATA_PATH+'validation_set','validation_set','Solution/validation_set')
TEST_DIR=_find(DATA_PATH+'testing_set',DATA_PATH+'test_set','test_set','testing_set','Solution/test_set')
print('train:',TRAIN_DIR,'| val:',VAL_DIR,'| test:',TEST_DIR)

In [ ]:
# 50x181 -> 56x192(÷8 패딩) -> U-Net -> 50x181 크롭
PAD=(5,6,3,3); RC=slice(3,53); CC=slice(5,186)
padf=lambda x: F.pad(x,PAD); cropf=lambda x: x[...,RC,CC]
def load_img(d):
    img=d[:6].float(); return (img-img.mean((1,2),keepdim=True))/(img.std((1,2),keepdim=True)+1e-6)

class RadarDS(Dataset):
    def __init__(s,paths,lbl=True): s.p=paths; s.lbl=lbl
    def __len__(s): return len(s.p)
    def __getitem__(s,i):
        d=torch.load(s.p[i],weights_only=True)
        if s.lbl: return load_img(d),(d[6]+1).long(),os.path.basename(s.p[i])  # 라벨 -1..3 -> 0..4
        return load_img(d),0,os.path.basename(s.p[i])

In [ ]:
class DC(nn.Module):
    def __init__(s,ci,co):
        super().__init__()
        s.n=nn.Sequential(nn.Conv2d(ci,co,3,padding=1),nn.BatchNorm2d(co),nn.ReLU(True),
                          nn.Conv2d(co,co,3,padding=1),nn.BatchNorm2d(co),nn.ReLU(True))
    def forward(s,x): return s.n(x)

class UNet(nn.Module):
    def __init__(s,cin=6,cout=5,b=64):
        super().__init__()
        s.e1=DC(cin,b); s.e2=DC(b,b*2); s.e3=DC(b*2,b*4); s.bo=DC(b*4,b*8); s.pool=nn.MaxPool2d(2)
        s.u3=nn.ConvTranspose2d(b*8,b*4,2,stride=2); s.d3=DC(b*8,b*4)
        s.u2=nn.ConvTranspose2d(b*4,b*2,2,stride=2); s.d2=DC(b*4,b*2)
        s.u1=nn.ConvTranspose2d(b*2,b,2,stride=2);   s.d1=DC(b*2,b); s.o=nn.Conv2d(b,cout,1)
    def forward(s,x):
        e1=s.e1(x); e2=s.e2(s.pool(e1)); e3=s.e3(s.pool(e2)); bo=s.bo(s.pool(e3))
        d3=s.d3(torch.cat([s.u3(bo),e3],1)); d2=s.d2(torch.cat([s.u2(d3),e2],1)); d1=s.d1(torch.cat([s.u1(d2),e1],1))
        return s.o(d1)

In [ ]:
# 학습: 가중 CE(배경 1, 비배경 50) — 채점 가중과 일치 (핵심 개선)
EPOCHS=10; BS=16
paths=sorted(glob.glob(f'{TRAIN_DIR}/*.mat.pt'))
tr,_=train_test_split(paths,test_size=0.1,random_state=42)
trdl=DataLoader(RadarDS(tr),batch_size=BS,shuffle=True,num_workers=4,drop_last=True)
model=UNet().to(DEV)
crit=nn.CrossEntropyLoss(weight=torch.tensor([1.,50.,50.,50.,50.],device=DEV))
opt=torch.optim.Adam(model.parameters(),lr=1e-3)
sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,EPOCHS); scaler=torch.amp.GradScaler('cuda')
for ep in range(EPOCHS):
    model.train(); tot=0
    for img,lbl,_ in trdl:
        img,lbl=img.to(DEV),lbl.to(DEV); opt.zero_grad()
        with torch.amp.autocast('cuda'): loss=crit(cropf(model(padf(img))),lbl)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); tot+=loss.item()
    sched.step(); print(f'epoch {ep+1}/{EPOCHS}  loss {tot/len(trdl):.4f}')

In [ ]:
# 예측 -> submission_val.csv / submission_test.csv (클래스 0..4 -> 라벨 -1..3)
def predict_csv(d,out):
    model.eval(); fns=[]; preds=[]
    dl=DataLoader(RadarDS(sorted(glob.glob(f'{d}/*.mat.pt')),lbl=False),batch_size=BS,num_workers=4)
    with torch.no_grad():
        for img,_,fn in dl:
            with torch.amp.autocast('cuda'): o=cropf(model(padf(img.to(DEV))))
            p=(o.argmax(1)-1).cpu().numpy().astype(int)
            for j in range(p.shape[0]): preds.append(p[j].flatten()); fns.append(fn[j])
    arr=np.stack(preds); df=pd.DataFrame(arr,columns=[f'pixel_{i}' for i in range(arr.shape[1])])
    df.insert(0,'filename',fns); df.to_csv(out,index=False); print('saved',out,df.shape)
predict_csv(VAL_DIR,'submission_val.csv')
predict_csv(TEST_DIR,'submission_test.csv')

In [ ]:
# 제출 zip
import zipfile
with zipfile.ZipFile('submission.zip','w') as z:
    z.write('submission_val.csv'); z.write('submission_test.csv')
print('submission.zip created')

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission_val.csv', 'submission_test.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)